In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------- DATSET #1: Insurance Prices ----------
# Analyzing medical insurance prices across regions  https://www.kaggle.com/code/kellibelcher/predicting-insurance-prices-across-regions/input
df = pd.read_csv("insurance.csv")
# I'm making my "Smoker" column the dependent variable in order to do logistic regression. Normally, I would want to predict "Charges", 
# but since that is a continus variable, it wouldn't work well for logistic regression.
y = df['smoker'].map({'yes': 1, 'no': 0})  # smoker = 1, non-smoker = 0

X = pd.get_dummies(df.drop(columns=['smoker']), drop_first=True).astype(float)  
print(y.value_counts())  # check the balance of smokers vs non-smokers

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # stratify keeps the same smoker ratio in both sets

# Scale the features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test) 

# Fit logistic regression ----------
logit = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)  # fit the model
pred = logit.predict(X_test_s)  # predicted class (0 or 1)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))  # precision, recall, f1 per class
print(confusion_matrix(y_test, pred))  # rows = actual, columns = predicted

coefs = pd.DataFrame({'feature': X.columns, 'coefficient': logit.coef_[0]}).sort_values('coefficient', key=abs, ascending=False)
print(coefs)

smoker
0    1064
1     274
Name: count, dtype: int64
Accuracy: 0.9664179104477612
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       213
           1       0.91      0.93      0.92        55

    accuracy                           0.97       268
   macro avg       0.95      0.95      0.95       268
weighted avg       0.97      0.97      0.97       268

[[208   5]
 [  4  51]]
            feature  coefficient
3           charges     3.858528
1               bmi    -1.644932
0               age    -1.187838
6  region_southeast     0.329840
4          sex_male     0.227697
2          children    -0.220353
5  region_northwest     0.049190
7  region_southwest     0.003762


Here is how I would interpret the results: 
Accuracy: 96.6%. Out of all 268 people in the test set, the model correctly identified whether they were a smoker or not 96.6% of the time.
Precision for smokers: 91%. Out of the 56 people it labeled "smoker" (51 correct + 5 wrong), 51 really were smokers. The other 5 were false alarms.
Recall (for smokers): 93%. Out of the 55 real smokers in the test set, it correctly identified 51 and missed 4.

Based off the above measurements, the model seems very accurate in predicting smoker vs non smoker. Insurance chargers clearly have the biggest impact in predicting smoker vs non smoker. 

Its important to note that there are 1,064 non-smokers vs. 274 smokers (about 80/20 split) in the dataset. This imbalance could cause the accuracy to be misleading.

In [5]:

# ---------- DATSET #2: Insurance Fruad ----------
# Analyzing possible fraudulent insurance claims https://data.mendeley.com/datasets/992mh7dk9y/2
df2 = pd.read_csv("insurance_claims.csv") 
df2 = df2.replace('?', 'Unknown')  # missing values are coded as "?"
# Fraud Yes/No fits well as a dependent variable for logistic regression.
df2['fraud'] = df2['fraud_reported'].map({'Y': 1, 'N': 0})  # fraud = 1, not fraud = 0

num_cols = ['age', 'months_as_customer', 'policy_annual_premium', 'number_of_vehicles_involved', 'total_claim_amount']
cat_cols = ['incident_severity', 'collision_type', 'insured_sex', 'property_damage']
df2 = df2[num_cols + cat_cols + ['fraud']].dropna()
y2 = df2['fraud']

# Make Categorical Variables into Binary Variables
X2 = pd.get_dummies(df2.drop(columns='fraud'), columns=cat_cols, drop_first=True).astype(float)  # dummies
print(y2.value_counts())  # checking the balance of fraud vs not fraud

# Train/test split
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

# Scale the features
scaler2 = StandardScaler()  # new scaler for this new X
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s = scaler2.transform(X2_test)

# Fit logistic regression
logit2 = LogisticRegression(max_iter=1000).fit(X2_train_s, y2_train)
pred2 = logit2.predict(X2_test_s)

print("Accuracy:", accuracy_score(y2_test, pred2))
print(classification_report(y2_test, pred2))
print(confusion_matrix(y2_test, pred2))

coefs2 = pd.DataFrame({'feature': X2.columns, 'coefficient': logit2.coef_[0]}).sort_values('coefficient', key=abs, ascending=False)
print(coefs2)

fraud
0    753
1    247
Name: count, dtype: int64
Accuracy: 0.825
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       151
           1       0.63      0.67      0.65        49

    accuracy                           0.82       200
   macro avg       0.76      0.77      0.77       200
weighted avg       0.83      0.82      0.83       200

[[132  19]
 [ 16  33]]
                             feature  coefficient
5     incident_severity_Minor Damage    -1.258934
6       incident_severity_Total Loss    -0.954382
7   incident_severity_Trivial Damage    -0.872778
1                 months_as_customer     0.253649
0                                age    -0.206071
10            collision_type_Unknown     0.159770
12           property_damage_Unknown     0.157224
4                 total_claim_amount     0.134319
3        number_of_vehicles_involved    -0.119535
13               property_damage_YES     0.099188
8      collision_type_Rear Collisi

Here is how I would interpret the results: 
Accuracy: 82.5% out of 200 claims in the test set, the model correctly classified 165.
Precision for fraudulent claims: 63% of every claim the model flagged as fraud, 63% actually were fraud.
Recall for fraudulent claims: 67% of all 49 real fraud cases, the model caught 33 (67%) and missed 16.

Precision for non-fraudulent claims: 89% of every claim the model predicted as not fraud, were actually not fraud.
Recall for non-fraudulent claims: Of every claim that actually wasn't fraud, the model correctly identified 87% of them.

The accuracy for this model is pretty strong at 82.5%. However, the precision and recall for fraudulent claims were both substantially lower.
The recall is the stat I care most about, because that says how many fraud claims were actually caught by the model. For fraud,
you want to minimize type 2 errors. I would rather spend time reviewing a case that terms out to not be fraud than miss a fraudulent one.
Fraud is very expensive!
From the coefficients, you can see that incident_severity_Minor Damage has a strong negative relationship with fraud, proving that minor claims are
usually less likely to be fraudulent. Frauds are after the higher damage claim amounts!